In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 1. 自动寻找文件路径
# =========================

possible_clean_paths = [
    Path("stage4_model_data_clean.parquet"),
    Path("stage4_outputs/stage4_model_data_clean.parquet"),
    Path("outputs/stage4_model_data_clean.parquet"),
]

possible_panel_paths = [
    Path("stage4_model_panel.parquet"),
    Path("stage4_outputs/stage4_model_panel.parquet"),
    Path("outputs/stage4_model_panel.parquet"),
]

clean_path = next((p for p in possible_clean_paths if p.exists()), None)
panel_path = next((p for p in possible_panel_paths if p.exists()), None)

if clean_path is None:
    raise FileNotFoundError("找不到 stage4_model_data_clean.parquet，请检查文件是否在当前目录、stage4_outputs 或 outputs 里面。")

if panel_path is None:
    raise FileNotFoundError("找不到 stage4_model_panel.parquet，请检查文件是否在当前目录、stage4_outputs 或 outputs 里面。")

print("Clean file:", clean_path)
print("Panel file:", panel_path)

df_clean = pd.read_parquet(clean_path)
df_panel = pd.read_parquet(panel_path)

print("df_clean shape:", df_clean.shape)
print("df_panel shape:", df_panel.shape)


# =========================
# 2. 统一 date 格式
# =========================

df_clean["date"] = pd.to_datetime(df_clean["date"])
df_panel["date"] = pd.to_datetime(df_panel["date"])


# =========================
# 3. 确定 merge key
# =========================
# 最推荐用 date + instrument_id
# 如果你的文件没有 instrument_id，再退而用 date + ticker

if "instrument_id" in df_clean.columns and "instrument_id" in df_panel.columns:
    merge_keys = ["date", "instrument_id"]
elif "ticker" in df_clean.columns and "ticker" in df_panel.columns:
    merge_keys = ["date", "ticker"]
else:
    raise ValueError("无法找到合并键。需要 date + instrument_id，或者 date + ticker。")

print("Merge keys:", merge_keys)


# =========================
# 4. 从 panel 里面选择要补充的基础变量
# =========================
# 注意：这里只补充 clean 里面没有的列，避免重复列名

candidate_cols_from_panel = [
    # volatility / liquidity / size
    "price_for_filter",
    "adv20",
    "ann_vol60",
    "year_start_market_cap",
    "market_cap_rank",

    # short-interest
    "dsi",
    "dtcn",
    "ddtcn",

    # calendar
    "year",
    "month",
    "day_of_week",
    "is_month_end",
    "is_quarter_end",

    # cost
    "impact_bps_at_participation_cap",
]

cols_to_add = [
    c for c in candidate_cols_from_panel
    if c in df_panel.columns and c not in df_clean.columns
]

print("Columns to add from panel:")
print(cols_to_add)

if len(cols_to_add) > 0:
    panel_small = df_panel[merge_keys + cols_to_add].drop_duplicates(subset=merge_keys).copy()

    rows_before = len(df_clean)

    df = df_clean.merge(
        panel_small,
        on=merge_keys,
        how="left"
    )

    rows_after = len(df)

    if rows_before != rows_after:
        raise ValueError(
            f"合并后行数变化了！合并前 {rows_before} 行，合并后 {rows_after} 行。"
            "说明 panel 里面 merge key 可能有重复。"
        )

else:
    print("clean 文件里已经有这些基础变量，不需要从 panel 额外合并。")
    df = df_clean.copy()

print("Merged df shape:", df.shape)


# =========================
# 5. 按股票和日期排序
# =========================

sort_keys = []

if "instrument_id" in df.columns:
    sort_keys.append("instrument_id")
elif "ticker" in df.columns:
    sort_keys.append("ticker")

sort_keys.append("date")

df = df.sort_values(sort_keys).copy()


# =========================
# 6. 生成 return reversal / momentum 特征
# =========================

if "r_on_today" in df.columns:
    df["on_reversal_1d"] = -df["r_on_today"]

if "r_id_lag1" in df.columns:
    df["id_reversal_1d"] = -df["r_id_lag1"]

if "r_cc_lag1" in df.columns:
    df["cc_reversal_1d"] = -df["r_cc_lag1"]

if "r_on_mean_5" in df.columns and "r_on_mean_20" in df.columns:
    df["on_mom_5_20"] = df["r_on_mean_5"] - df["r_on_mean_20"]
    df["gap_vs_on_mean20"] = df["r_on_today"] - df["r_on_mean_20"]

if "r_id_mean_5_lag1" in df.columns and "r_id_mean_20_lag1" in df.columns:
    df["id_mom_5_20"] = df["r_id_mean_5_lag1"] - df["r_id_mean_20_lag1"]

if "r_cc_mean_5_lag1" in df.columns and "r_cc_mean_20_lag1" in df.columns:
    df["cc_mom_5_20"] = df["r_cc_mean_5_lag1"] - df["r_cc_mean_20_lag1"]


# =========================
# 7. 生成风险调整后的收益特征
# =========================

eps = 1e-8

if "r_on_today" in df.columns and "vol_20_lag1" in df.columns:
    df["r_on_today_to_vol20"] = df["r_on_today"] / (df["vol_20_lag1"] + eps)

if "r_cc_lag1" in df.columns and "vol_20_lag1" in df.columns:
    df["r_cc_lag1_to_vol20"] = df["r_cc_lag1"] / (df["vol_20_lag1"] + eps)

if "abs_r_on_today" in df.columns and "vol_20_lag1" in df.columns:
    df["abs_r_on_today_to_vol20"] = df["abs_r_on_today"] / (df["vol_20_lag1"] + eps)


# =========================
# 8. 生成 liquidity / size 的 log 特征
# =========================

if "adv20" in df.columns:
    df["log_adv20"] = np.log1p(df["adv20"].clip(lower=0))

if "year_start_market_cap" in df.columns:
    df["log_market_cap"] = np.log1p(df["year_start_market_cap"].clip(lower=0))

if "price_for_filter" in df.columns:
    df["log_price_for_filter"] = np.log1p(df["price_for_filter"].clip(lower=0))

if "impact_bps_at_participation_cap" in df.columns:
    df["log_impact_bps"] = np.log1p(df["impact_bps_at_participation_cap"].clip(lower=0))


# =========================
# 9. 生成 short-interest 的 lag 和变化量特征
# =========================
# 这里用 groupby 对每只股票分别 shift，避免不同股票之间串数据

group_key = "instrument_id" if "instrument_id" in df.columns else "ticker"

for col in ["dsi", "dtcn", "ddtcn"]:
    if col in df.columns:
        df[f"{col}_lag1"] = df.groupby(group_key)[col].shift(1)
        df[f"{col}_lag5"] = df.groupby(group_key)[col].shift(5)
        df[f"{col}_lag20"] = df.groupby(group_key)[col].shift(20)

        df[f"{col}_change_5"] = df[col] - df[f"{col}_lag5"]
        df[f"{col}_change_20"] = df[col] - df[f"{col}_lag20"]

        # 更保守的版本：只用到 lag1 之前的信息
        df[f"{col}_change_5_lag1"] = df[f"{col}_lag1"] - df.groupby(group_key)[col].shift(6)
        df[f"{col}_change_20_lag1"] = df[f"{col}_lag1"] - df.groupby(group_key)[col].shift(21)


# =========================
# 10. 生成更细的 calendar 特征
# =========================

df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek

df["is_monday"] = (df["day_of_week"] == 0).astype(int)
df["is_friday"] = (df["day_of_week"] == 4).astype(int)

df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

df["is_quarter_start"] = df["date"].dt.is_quarter_start.astype(int)
df["is_quarter_end"] = df["date"].dt.is_quarter_end.astype(int)

df["is_year_start"] = df["date"].dt.is_year_start.astype(int)
df["is_year_end"] = df["date"].dt.is_year_end.astype(int)


# =========================
# 11. 生成横截面 rank 和 z-score 特征
# =========================
# 每一天内部，对所有股票做排名和标准化
# 这一步非常重要，因为你的任务是每天给股票排序

cs_base_cols = [
    # return
    "r_on_today",
    "r_on_lag1",
    "r_id_lag1",
    "r_cc_lag1",
    "r_on_mean_5",
    "r_on_mean_20",
    "r_id_mean_5_lag1",
    "r_id_mean_20_lag1",
    "r_cc_mean_5_lag1",
    "r_cc_mean_20_lag1",

    # new return features
    "on_reversal_1d",
    "id_reversal_1d",
    "cc_reversal_1d",
    "on_mom_5_20",
    "id_mom_5_20",
    "cc_mom_5_20",
    "gap_vs_on_mean20",

    # risk
    "vol_20_lag1",
    "vol_60_lag1",
    "ann_vol60",
    "abs_r_on_today",
    "abs_r_cc_lag1",
    "r_on_today_to_vol20",
    "r_cc_lag1_to_vol20",
    "abs_r_on_today_to_vol20",

    # liquidity / size
    "price_for_filter",
    "adv20",
    "log_adv20",
    "year_start_market_cap",
    "log_market_cap",
    "market_cap_rank",
    "impact_bps_at_participation_cap",
    "log_impact_bps",

    # short interest
    "dsi",
    "dtcn",
    "ddtcn",
    "dsi_lag1",
    "dtcn_lag1",
    "ddtcn_lag1",
    "dsi_change_5",
    "dsi_change_20",
    "dtcn_change_5",
    "dtcn_change_20",
    "ddtcn_change_5",
    "ddtcn_change_20",
]

cs_base_cols = [c for c in cs_base_cols if c in df.columns]

for col in cs_base_cols:
    # 横截面百分位排名，范围大致是 0 到 1
    df[f"{col}_cs_rank"] = df.groupby("date")[col].rank(pct=True)

    # 横截面 z-score
    daily_mean = df.groupby("date")[col].transform("mean")
    daily_std = df.groupby("date")[col].transform("std")

    df[f"{col}_cs_z"] = (df[col] - daily_mean) / daily_std.replace(0, np.nan)


# =========================
# 12. 简单检查缺失值情况
# =========================

print("\nTop columns with missing values:")
missing_ratio = df.isna().mean().sort_values(ascending=False)
print(missing_ratio.head(30))


# =========================
# 13. 保存新文件
# =========================

output_dir = Path("stage4_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "stage4_model_data_clean_plus_features.parquet"

df.to_parquet(output_path, index=False)

print("\nDone.")
print("Saved to:", output_path)
print("Final shape:", df.shape)

print("\nFinal columns:")
for c in df.columns:
    print(c)

Clean file: stage4_model_data_clean.parquet
Panel file: stage4_model_panel.parquet
df_clean shape: (3773971, 35)
df_panel shape: (3773971, 55)
Merge keys: ['date', 'instrument_id']
Columns to add from panel:
['impact_bps_at_participation_cap']
Merged df shape: (3773971, 36)


C:\Users\yuchao\AppData\Local\Temp\ipykernel_12560\1711603047.py:314: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_cs_z"] = (df[col] - daily_mean) / daily_std.replace(0, np.nan)
C:\Users\yuchao\AppData\Local\Temp\ipykernel_12560\1711603047.py:308: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_cs_rank"] = df.groupby("date")[col].rank(pct=True)
C:\Users\yuchao\AppData\Local\Temp\ipykernel_12560\1711603047.py:314: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram


Top columns with missing values:
log_impact_bps_cs_z                     1.000000
impact_bps_at_participation_cap_cs_z    1.000000
dsi_change_5_cs_z                       0.524914
dtcn_change_5_cs_z                      0.524650
ddtcn_change_5_cs_z                     0.524650
ann_vol60_cs_z                          0.010599
ddtcn_change_20_cs_z                    0.008839
dsi_change_20_cs_z                      0.008839
dtcn_change_20_cs_z                     0.008839
dtcn_change_20_lag1                     0.007890
dsi_change_20_lag1                      0.007890
ddtcn_change_20_lag1                    0.007890
ddtcn_change_20                         0.007515
dsi_change_20                           0.007515
dtcn_lag20                              0.007515
dsi_lag20                               0.007515
dtcn_change_20_cs_rank                  0.007515
dsi_change_20_cs_rank                   0.007515
ddtcn_change_20_cs_rank                 0.007515
ddtcn_lag20                        